# Zero-Shot GPT-4o Baseline (No RAG)

This notebook implements a **zero-shot baseline** for comparison with RAG-based systems.

**Approach:**
- Direct GPT-4o prompting
- Input: **Transcript only** (no patient profile, no retrieval, no previous history)
- No external knowledge

**Purpose:**
- Establish pure baseline performance
- Isolate the value added by patient profiles and RAG systems
- Address reviewer questions about simpler baselines


In [ ]:
# ============================================================================
# CELL 1: CONFIGURATION & IMPORTS
# ============================================================================

import re, json, time
import os
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime

import pandas as pd
import tiktoken
import openai

# =========================
# Path Configuration
# =========================

TRANSCRIPT_ROOT = Path("Transcripts_Whisper_order by patient")
PATIENT_PROFILE_CSV = Path("patient_profiles copy.csv")
FORM_ROOT = Path("data/Visiting Form/Visiting Form - Order by Patient")

OUT_DIR = Path("zero_shot_baseline_output_whisper")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# API Configuration
# =========================

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# =========================
# Model Configuration
# =========================

MODEL_SUMMARY = "gpt-4o"           
TEMP_SUMMARY = 0.3                  

# =========================
# Official Form Template
# =========================

OFFICIAL_FORM_TEMPLATE = """
1.【利用者の状態（病状・トピックス）】
このセクションは以下の内容で構成されています。

- トピックス: 
  今回の診察で最も重要な臨床観察について記述してください。
判断基準：
  1. 安全リスク: 転倒、出血、強い痛み、発熱（38℃以上）、意識変容、呼吸困難
  2. 症状の変化: 新たな症状、既存症状の悪化・改善、処置の変更
  3. 日常生活への影響: ADL低下、活動制限、睡眠障害
  4. 変化なし: 上記が該当しない場合のみ「特に重大な問題なし」
  
  【記載方法】
  - 具体的に記載
  - 時系列を明記: 「いつ」「何が」「どう変化したか」を含める
  - 臨床的意義: 日常生活や今後のケアへの影響を示唆

- フォームから抽出した各カテゴリーを【..】内にリストアップしてください。各カテゴリごとに、観察した事実を**臨床的文脈**と共に説明します。
  【記載ルール】
  1. 単なる事実の羅列ではなく、臨床的に意味のある文章で記載
  2. 複数の観察を関連付けて記載（例: 睡眠と疼痛の関係）
  3. 時系列や因果関係があれば明記
  4. 患者の主観的訴えと客観的観察を区別

- <処置内容・訓練内容>: 会話に基づき、実施した処置や訓練の内容を記載。
- <備考>: 特別な注意事項や重要な補足情報を記載。
- <外部連絡・伝達事項>: 外部機関（医師、ケアマネージャー、家族など）への連絡事項を記載。
"""

print("✅ Configuration loaded!")
print(f"📂 Output directory: {OUT_DIR}")
print(f"🤖 Model: Summary={MODEL_SUMMARY}")
print(f"🎯 Focus: SUMMARIZATION ONLY (No extraction)")
print(f"📝 Input: TRANSCRIPT ONLY (No patient profile)")


In [ ]:
# 🔍 Diagnostic: Check file paths and directory structure
print("=" * 60)
print("FILE PATH DIAGNOSTICS")
print("=" * 60)

# Check FORM_ROOT
print(f"\n📂 FORM_ROOT: {FORM_ROOT}")
print(f"   Exists: {FORM_ROOT.exists()}")

if FORM_ROOT.exists():
    print(f"\n📁 Contents of FORM_ROOT:")
    try:
        contents = list(FORM_ROOT.iterdir())
        print(f"   Found {len(contents)} items")
        for item in contents[:10]:  # Show first 10 items
            print(f"   - {item.name} {'(dir)' if item.is_dir() else '(file)'}")
        if len(contents) > 10:
            print(f"   ... and {len(contents) - 10} more items")
    except Exception as e:
        print(f"   ⚠️ Error reading directory: {e}")
    
    # Test different glob patterns for EXCEL files
    print(f"\n🔍 Testing glob patterns for .xlsx files:")
    patterns = [
        "*/*.xlsx",
        "*.xlsx", 
        "*/*",
        "**/*.xlsx"
    ]
    
    for pattern in patterns:
        matches = list(FORM_ROOT.glob(pattern))
        print(f"   Pattern '{pattern}': {len(matches)} files")
        if matches and pattern.endswith('.xlsx'):
            print(f"      Example: {matches[0].relative_to(FORM_ROOT)}")

# Check TRANSCRIPT_ROOT
print(f"\n📂 TRANSCRIPT_ROOT: {TRANSCRIPT_ROOT}")
print(f"   Exists: {TRANSCRIPT_ROOT.exists()}")

# Check patient profile CSV
print(f"\n📂 PATIENT_PROFILE_CSV: {PATIENT_PROFILE_CSV}")
print(f"   Exists: {PATIENT_PROFILE_CSV.exists()}")

if PATIENT_PROFILE_CSV.exists():
    try:
        profiles_df = pd.read_csv(PATIENT_PROFILE_CSV)
        print(f"   ✅ Loaded {len(profiles_df)} patient profiles")
    except Exception as e:
        print(f"   ⚠️ Error loading: {e}")

print("\n" + "=" * 60)

In [ ]:
# ============================================================================
# CELL 2: PROCESSING METER
# ============================================================================

class ProcessingMeter:
    """Track processing metrics, API calls, tokens, and costs"""
    
    def __init__(self):
        self.start_time = None
        self.end_time = None
        self.api_calls = {'summary': 0, 'total': 0}
        self.tokens_used = {'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0}
        self.costs = {'summary_cost': 0.0, 'total_cost': 0.0}
        
        # OpenAI pricing (as of 2024)
        self.pricing = {
            'gpt-4o': {'input': 2.50, 'output': 10.00},  # per 1M tokens
        }
    
    def start_timer(self):
        """Start timing the process"""
        self.start_time = time.time()
        print(f"🕐 Processing started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    def end_timer(self):
        """End timing and calculate duration"""
        self.end_time = time.time()
        duration = self.end_time - self.start_time
        print(f"⏰ Processing completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⌛ Total duration: {self.format_duration(duration)}")
        return duration
    
    def format_duration(self, seconds):
        """Format duration in readable format"""
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        return f"{hours}h {minutes}m {secs}s"
    
    def track_api_call(self, call_type, model, input_tokens=0, output_tokens=0):
        """Track an API call and calculate cost"""
        self.api_calls[call_type] += 1
        self.api_calls['total'] += 1
        
        # Track tokens
        self.tokens_used['input_tokens'] += input_tokens
        self.tokens_used['output_tokens'] += output_tokens
        self.tokens_used['total_tokens'] += input_tokens + output_tokens
        
        # Calculate cost
        cost = 0.0
        if model in self.pricing:
            cost = (input_tokens / 1_000_000) * self.pricing[model]['input'] + \
                   (output_tokens / 1_000_000) * self.pricing[model]['output']
        
        # Add to appropriate cost category
        self.costs[f'{call_type}_cost'] += cost
        self.costs['total_cost'] += cost
        
        return cost
    
    def print_summary(self):
        """Print detailed summary of processing metrics"""
        duration = self.end_time - self.start_time if self.end_time else 0
        
        print(f"\n{'='*60}")
        print(f"📊 ZERO-SHOT BASELINE PROCESSING SUMMARY")
        print(f"{'='*60}")
        
        print(f"⏱️  TIMING:")
        print(f"   • Duration: {self.format_duration(duration)}")
        
        print(f"\n📞 API CALLS:")
        for call_type, count in self.api_calls.items():
            print(f"   • {call_type.title()}: {count}")
        
        print(f"\n🔤 TOKENS USED:")
        print(f"   • Input tokens: {self.tokens_used['input_tokens']:,}")
        print(f"   • Output tokens: {self.tokens_used['output_tokens']:,}")
        print(f"   • Total tokens: {self.tokens_used['total_tokens']:,}")
        
        print(f"\n💰 COSTS:")
        for cost_type, amount in self.costs.items():
            print(f"   • {cost_type.replace('_', ' ').title()}: ${amount:.4f}")
        
        print(f"{'='*60}")

# Global meter instance
processing_meter = ProcessingMeter()

print("✅ Processing meter ready!")


In [ ]:
# ============================================================================
# CELL 3: ZERO-SHOT SUMMARY GENERATION
# ============================================================================

class ZeroShotSummarizer:
    """Zero-shot summary generation - transcript only, no external knowledge"""
    
    def __init__(self, api_key: str, meter: 'ProcessingMeter'):
        self.client = openai.AsyncOpenAI(api_key=api_key, timeout=60.0, max_retries=3)
        self.meter = meter
        self.system_message = f"""あなたは経験豊富な訪問看護師です。以下の公式様式に従って、専門的で正確な看護記録を作成してください。

【公式記録様式】
{OFFICIAL_FORM_TEMPLATE}

【必須作成ルール】
1. 公式様式のすべてのセクションを必ず含めること（セクションの省略は厳禁）
2. 今回の訪問会話を最優先とし、推測や創作は禁止
3. 数値・日付・時間は会話に明記されている場合のみ記載

【記載の指針】
• 会話に出た具体的な食べ物、薬剤名、場所、時間を必ず記載
• 患者の発言や認識も重要な情報として記載
• 会話で言及がない項目は「該当なし」または「特記事項なし」
• 推測・創作・抽象的表現は一切禁止

【出力形式 - 厳守】
• 公式フォームのすべてのセクションを、構成や見出しを変更せずに出力してください。
"""
    
    async def generate_summary(self, transcript: str, categories: List[str]) -> str:
        """Generate summary using ONLY transcript (no patient profile, no RAG)"""
        
        # Inject トピックス category
        enhanced_categories = list(categories) if categories else []
        if "トピックス" not in enhanced_categories:
            enhanced_categories.insert(0, "トピックス")
        
        cats_lines = "\n".join(f"【{c}】" for c in enhanced_categories)
        
        # Build user prompt - ONLY transcript (NO patient profile)
        user_prompt = f"""【現在の訪問記録】
{transcript}

動的カテゴリ -（以下の順序で必ず全て記載）
{cats_lines}

"""
        
        # Count tokens
        try:
            encoding = tiktoken.encoding_for_model(MODEL_SUMMARY)
        except Exception:
            encoding = tiktoken.get_encoding("o200k_base")
        
        system_tokens = len(encoding.encode(self.system_message))
        user_tokens = len(encoding.encode(user_prompt))
        input_tokens = system_tokens + user_tokens
        
        # API call
        response = await self.client.chat.completions.create(
            model=MODEL_SUMMARY,
            messages=[
                {"role": "system", "content": self.system_message},
                {"role": "user", "content": user_prompt}
            ],
            temperature=TEMP_SUMMARY,
        )
        
        # Track API call to the meter instance
        output_tokens = response.usage.completion_tokens if response.usage else 0
        actual_input_tokens = response.usage.prompt_tokens if response.usage else input_tokens
        self.meter.track_api_call('summary', MODEL_SUMMARY, actual_input_tokens, output_tokens)
        
        return response.choices[0].message.content

print("✅ Zero-shot summarizer loaded (transcript only - no patient profile)!")

In [ ]:
# ============================================================================
# CELL 5: FORM PROCESSING & PIPELINE
# ============================================================================

import re


def load_form_sheets(form_path: Path) -> Dict[str, pd.DataFrame]:
    """Load all sheets from an Excel form file"""
    try:
        sheets = pd.read_excel(form_path, sheet_name=None, header=None)
        print(f"✅ Loaded {len(sheets)} sheets from {form_path.name}")
        return sheets
    except Exception as e:
        print(f"❌ Error loading Excel file {form_path}: {e}")
        return {}


def extract_category_names(sheet_df: pd.DataFrame,
                          section_header: str = "利用者の状態（病状・トピックス）") -> List[str]:
    """Extract category names from form sheet"""
    if sheet_df is None or sheet_df.empty:
        return []

    # Locate section row
    idxs = sheet_df[
        sheet_df.apply(lambda row: row.astype(str).str.contains(section_header, na=False).any(), axis=1)
    ].index

    if len(idxs) == 0:
        print(f"⚠️ Section header '{section_header}' not found in sheet")
        return []

    start = idxs[0]
    cat_pat = r"【(.*?)】"
    cats: List[str] = []

    # Look for categories below the header
    for r in range(start + 1, sheet_df.shape[0]):
        row = sheet_df.iloc[r]
        if row.isna().all():
            break
        text = " ".join(row.dropna().astype(str))
        cats.extend(re.findall(cat_pat, text))

    # Normalize & dedupe
    seen = set()
    out = []
    for c in cats:
        c = c.strip()
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    
    print(f"🎯 Extracted {len(out)} categories: {out}")
    return out


def map_form_to_transcript_path(form_path: Path, form_root: Path, transcript_root: Path) -> Path:
    """Map a form Excel file to its corresponding transcript file by matching filenames"""
    
    relative_path = form_path.relative_to(form_root)
    patient_folder = relative_path.parts[0] if relative_path.parts else ""
    
    # Extract patient ID
    patient_id = ""
    if patient_folder.startswith('P') and patient_folder[1:].isdigit():
        patient_id = patient_folder
    else:
        match = re.search(r'P(\d+)', form_path.name)
        if match:
            patient_id = f"P{match.group(1)}"
    
    if not patient_id:
        print(f"⚠️ Could not extract patient ID from {form_path}")
        return transcript_root / "unknown.txt"
    
    # Extract the base filename from form (without extension)
    form_stem = form_path.stem
    
    # Look for transcript with matching filename
    transcript_patient_dir = transcript_root / patient_id
    if transcript_patient_dir.exists():
        # Try to find exact match first
        matching_transcript = transcript_patient_dir / f"{form_stem}.txt"
        if matching_transcript.exists():
            return matching_transcript
        
        # Fallback: look for any txt file with similar name pattern
        txt_files = list(transcript_patient_dir.glob("*.txt"))
        for txt_file in txt_files:
            if txt_file.stem == form_stem:
                return txt_file
        
        # If no exact match found, use first file (backward compatibility)
        if txt_files:
            return txt_files[0]
    
    print(f"⚠️ No transcript found for patient {patient_id}")
    return transcript_root / patient_id / "not_found.txt"


async def process_all_forms_zero_shot(
    form_root: Path,
    transcript_root: Path,
    output_csv: str = "zero_shot_baseline_results.csv",
    limit: int = None
) -> pd.DataFrame:
    """Process all forms using zero-shot approach (transcript only)"""
    
    # Find all Excel form files
    form_files = sorted(form_root.glob("*/*.xlsx"))
    print(f"📄 Found {len(form_files)} form files")
    
    if limit:
        form_files = form_files[:limit]
        print(f"📊 Processing limited to first {limit} files")
    
    if not form_files:
        print("⚠️ No form files found! Check FORM_ROOT path.")
        return pd.DataFrame()
    
    # Initialize meter and summarizer
    meter = ProcessingMeter()
    meter.start_timer()
    
    summarizer = ZeroShotSummarizer(api_key=OPENAI_API_KEY, meter=meter)
    
    results = []
    
    for idx, form_path in enumerate(form_files, 1):
        print(f"\n{'='*60}")
        print(f"Processing {idx}/{len(form_files)}: {form_path.name}")
        print(f"{'='*60}")
        
        try:
            # Extract patient ID from path
            relative_path = form_path.relative_to(form_root)
            patient_folder = relative_path.parts[0] if relative_path.parts else ""
            
            patient_id = ""
            if patient_folder.startswith('P') and patient_folder[1:].isdigit():
                patient_id = patient_folder
            else:
                match = re.search(r'P(\d+)', form_path.name)
                if match:
                    patient_id = f"P{match.group(1)}"
            
            if not patient_id:
                print(f"⚠️ Could not extract patient ID from {form_path}")
                continue
            
            # Map to corresponding transcript
            transcript_path = map_form_to_transcript_path(form_path, form_root, transcript_root)
            
            if not transcript_path.exists():
                print(f"⚠️ Transcript not found: {transcript_path}")
                continue
            
            # Read transcript
            with open(transcript_path, 'r', encoding='utf-8') as f:
                transcript = f.read().strip()
            
            if not transcript:
                print(f"⚠️ Empty transcript for {patient_id}")
                continue
            
            print(f"✅ Loaded transcript: {len(transcript)} characters")
            
            # Extract categories from form (like AC-RAG does)
            sheets = load_form_sheets(form_path)
            sheet1 = sheets.get("Sheet1")
            if sheet1 is None and len(sheets) > 0:
                sheet1 = list(sheets.values())[0]
            
            categories = extract_category_names(sheet1, "利用者の状態（病状・トピックス）")
            
            if not categories:
                categories = ["睡眠", "疼痛", "活動", "服薬", "水分", "栄養"]
                print(f"⚠️ No categories found, using fallback: {categories}")
            
            # Generate summary (zero-shot with transcript only)
            summary = await summarizer.generate_summary(
                transcript=transcript,
                categories=categories
            )
            
            # Store result
            results.append({
                'Patient ID': patient_id,
                'Form File': form_path.name,
                'Transcript File': transcript_path.name,
                'Categories': ', '.join(categories),
                'Summary': summary
            })
            
            print(f"✅ Generated summary: {len(summary)} characters")
            print(f"📊 Cumulative: {meter.api_calls['total']} API calls, {meter.tokens_used['total_tokens']:,} tokens, ${meter.costs['total_cost']:.4f}")
            
        except Exception as e:
            print(f"❌ Error processing {form_path.name}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # End timer
    meter.end_timer()
    
    # Create DataFrame and save
    df_results = pd.DataFrame(results)
    
    if not df_results.empty:
        output_path = OUT_DIR / output_csv
        df_results.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"\n✅ Saved {len(df_results)} results to {output_path}")
    else:
        print("\n⚠️ No results to save")
    
    # Final meter report
    meter.print_summary()
    
    return df_results

print("✅ Form processing functions loaded!")
print("   • load_form_sheets() - Load Excel sheets")
print("   • extract_category_names() - Extract 【】categories from form")
print("   • map_form_to_transcript_path() - Match form to transcript")
print("   • process_all_forms_zero_shot() - Main pipeline")

In [ ]:
# ============================================================================
# CELL 6: RUN PROCESSING
# ============================================================================

# Run zero-shot baseline processing (SUMMARIZATION ONLY - TRANSCRIPT ONLY)
# Set limit=None to process all forms, or limit=10 for testing

results = await process_all_forms_zero_shot(
    form_root=FORM_ROOT,
    transcript_root=TRANSCRIPT_ROOT,
    output_csv="zero_shot_baseline_results.csv"
)

print("\n" + "="*80)
print("🎉 ZERO-SHOT BASELINE PROCESSING COMPLETE")
print("="*80)
print(f"\n📊 Processed {len(results)} summaries")
print(f"📂 Results saved to: {OUT_DIR}")
print(f"\n🎯 Input: Transcript only (no patient profile)")
print(f"📝 Method: Pure zero-shot prompting")